In [18]:
import os
import sys

from pyspark.sql import SparkSession


os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# spark = SparkSession.builder \
#     .appName("basic_app") \
#     .config("spark.driver.memory", "1g") \
#     .master("local[*]") \
#     .getOrCreate()

spark = SparkSession.builder.getOrCreate()

print(spark.version)

df = spark.createDataFrame(
    [(1, "a"), (2, "b"), (3, "c")],
    ("id", "value")
)

df.show()

4.2.0
+---+-----+
| id|value|
+---+-----+
|  1|    a|
|  2|    b|
|  3|    c|
+---+-----+



In [19]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("date", StringType(), False),
    StructField("product", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("quantity", IntegerType(), False)
])

df = spark.read.csv("sales.csv", schema=schema, header=True)
df.show()

+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
+---+----------+----------+------+--------+



In [20]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)



In [21]:
from pyspark.sql import functions as F

df = df.withColumn("total", F.col("price") * F.col("quantity"))
df.filter(F.col("total") < 1000).show()

+---+----------+----------+-----+--------+-----+
| id|      date|   product|price|quantity|total|
+---+----------+----------+-----+--------+-----+
|  2|2024-01-11|     Mouse| 25.0|       3| 75.0|
|  3|2024-01-12|  Keyboard| 45.0|       2| 90.0|
|  4|2024-01-13|   Monitor|300.0|       1|300.0|
|  6|2024-01-15|     Mouse| 20.0|       4| 80.0|
|  7|2024-01-16|   Monitor|290.0|       2|580.0|
|  8|2024-01-17|  Keyboard| 50.0|       1| 50.0|
|  9|2024-01-18|Headphones| 80.0|       5|400.0|
+---+----------+----------+-----+--------+-----+



In [22]:
from pyspark.sql import functions as F

df = df.withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
df = df.withColumn("weekday", F.date_format(F.col("date"), "EEEE"))
df = df.groupBy("weekday").agg(
    F.sum("total").alias("weekday_total"),
    F.listagg("product", " ").alias("products"),
    F.first("date")
)
df.show()

+---------+-------------+----------------+-----------+
|  weekday|weekday_total|        products|first(date)|
+---------+-------------+----------------+-----------+
|Wednesday|       1250.0| Laptop Keyboard| 2024-01-10|
|  Tuesday|        580.0|         Monitor| 2024-01-16|
|   Friday|       1390.0| Keyboard Laptop| 2024-01-12|
| Thursday|        475.0|Mouse Headphones| 2024-01-11|
| Saturday|        300.0|         Monitor| 2024-01-13|
|   Monday|         80.0|           Mouse| 2024-01-15|
|   Sunday|       1250.0|          Laptop| 2024-01-14|
+---------+-------------+----------------+-----------+



In [23]:
df_bad = spark.read.option("mode", "DROPMALFORMED").csv("sales_bad.csv", schema, ",", header=True)
df_bad.show(20)

+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
+---+----------+----------+------+--------+



In [24]:
df_bad = spark.read.option("mode", "PERMISSIVE").csv("sales_bad.csv", schema, ",", header=True)
df_bad.count()

11

In [25]:
df_bad = spark.read.option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record").csv("sales_bad.csv", schema=schema, sep=",", header=True)
df_bad.show(20)

+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
| 11|2024-01-20|  Portátil|  NULL|       2|
+---+----------+----------+------+--------+



In [26]:
from pyspark.sql import functions as F

schema_drop = StructType([
    StructField("id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
])

df_drop = (spark.read
           .option("header", True)
           .option("sep", ",")
           .option("mode", "DROPMALFORMED")
           .schema(schema_drop)
           .csv("sales_bad.csv"))

print("Count con DROPMALFORMED:", df_drop.filter(F.col("price").isNotNull()).count())
df_drop.show(20)

schema_perm = StructType([
    StructField("id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True)
])

df_permissive = (spark.read
                 .option("header", True)
                 .option("sep", ",")
                 .option("mode", "PERMISSIVE")
                 .option("columnNameOfCorruptRecord", "_corrupt_record")
                 .schema(schema_perm)
                 .csv("sales_bad.csv"))

print("Count con PERMISSIVE:", df_permissive.count())
df_permissive.filter(F.col("_corrupt_record").isNotNull()).show(truncate=False)
df_permissive.filter(F.col("_corrupt_record").isNull()).show(truncate=False)

Count con DROPMALFORMED: 10
+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
+---+----------+----------+------+--------+

Count con PERMISSIVE: 11
+---+----------+--------+-----+--------+-------------------------------------+
|id |date      |product |price|quantity|_corrupt_record                      |
+---+----------+--------+-----+--------+-------------------------------------+
|11 |2024-01-20|Portátil|NULL |2       |11,2024-01-20,Portátil,not_a_number,2|
+---+---------

In [27]:
from pyspark.sql import DataFrame
df_permissive.limit(5).show()

+---+----------+--------+------+--------+---------------+
| id|      date| product| price|quantity|_corrupt_record|
+---+----------+--------+------+--------+---------------+
|  1|2024-01-10|  Laptop|1200.0|       1|           NULL|
|  2|2024-01-11|   Mouse|  25.0|       3|           NULL|
|  3|2024-01-12|Keyboard|  45.0|       2|           NULL|
|  4|2024-01-13| Monitor| 300.0|       1|           NULL|
|  5|2024-01-14|  Laptop|1250.0|       1|           NULL|
+---+----------+--------+------+--------+---------------+



In [28]:
df = spark.read.csv("transactions.csv", header=True)
df.show(5)

+--------------+-----------+-------+------+-----------+----------+
|transaction_id|customer_id|country|amount|   category|      date|
+--------------+-----------+-------+------+-----------+----------+
|             1|        101|  Spain| 120.5|Electronics|2024-01-15|
|             2|        102| France|  75.0|    Fashion|2024-01-17|
|             3|        103|Germany| 210.3|       Home|2024-01-20|
|             4|        104|  Spain|  33.0|      Books|2024-02-02|
|             5|        105| France| 450.0|Electronics|2024-02-05|
+--------------+-----------+-------+------+-----------+----------+
only showing top 5 rows


In [29]:
from pyspark.sql.functions import col, year, month

df = df.withColumn("year", year(col("date")))
df = df.withColumn("month", month(col("date")))
(df.write
    .mode("overwrite")
    .partitionBy("year", "month")
    .option("compression", "snappy")
    .parquet("output/parquet/transactions"))

In [30]:
spark.read.parquet("output/parquet/transactions").filter((col("year") == 2024) & (col("month") == 1)).show()

+--------------+-----------+-------+------+-----------+----------+----+-----+
|transaction_id|customer_id|country|amount|   category|      date|year|month|
+--------------+-----------+-------+------+-----------+----------+----+-----+
|             1|        101|  Spain| 120.5|Electronics|2024-01-15|2024|    1|
|             2|        102| France|  75.0|    Fashion|2024-01-17|2024|    1|
|             3|        103|Germany| 210.3|       Home|2024-01-20|2024|    1|
+--------------+-----------+-------+------+-----------+----------+----+-----+



In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, col, expr, rand, floor

print("autoBroadcastJoinThreshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

lookup = spark.createDataFrame([(1, "A"), (2, "B"), (3, "C")], ["id", "label"])
big = spark.range(0, 2_000_000).withColumn("id", (col("id") % 3) + 1)

j1 = big.join(lookup, "id", "inner")
j1.explain(True)   # SortMergeJoin vs BroadcastHashJoin

j2 = big.join(broadcast(lookup), "id", "inner")
j2.explain(True)   # BroadcastHashJoin

n_salts = 8
big_salted = big.withColumn("salt", floor(rand() * n_salts))
lookup_salted = lookup.crossJoin(spark.range(0, n_salts).withColumnRenamed("id", "salt"))
joined_salted = big_salted.join(lookup_salted, ["id", "salt"])
joined_salted.explain(True)


autoBroadcastJoinThreshold: 10485760b
shuffle.partitions: 200
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [id])
:- Project [((id#929L % cast(3 as bigint)) + cast(1 as bigint)) AS id#930L]
:  +- Range (0, 2000000, step=1, splits=Some(16))
+- LogicalRDD [id#927L, label#928], false

== Analyzed Logical Plan ==
id: bigint, label: string
Project [id#930L, label#928]
+- Join Inner, (id#930L = id#927L)
   :- Project [((id#929L % cast(3 as bigint)) + cast(1 as bigint)) AS id#930L]
   :  +- Range (0, 2000000, step=1, splits=Some(16))
   +- LogicalRDD [id#927L, label#928], false

== Optimized Logical Plan ==
Project [id#930L, label#928]
+- Join Inner, (id#930L = id#927L)
   :- Project [((id#929L % 3) + 1) AS id#930L]
   :  +- Filter isnotnull(((id#929L % 3) + 1))
   :     +- Range (0, 2000000, step=1, splits=Some(16))
   +- Filter isnotnull(id#927L)
      +- LogicalRDD [id#927L, label#928], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#930L, label#928]
   +- 

In [32]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

data = [
    (1, "sales", "Alice", 70000, "2024-05-01"),
    (2, "sales", "Bob",   80000, "2024-06-01"),
    (3, "eng",   "Carol", 95000, "2024-05-21"),
    (4, "eng",   "Dave",  95000, "2024-06-01"),
    (5, "sales", "Eve",   80000, "2024-04-15"),
    (6, "eng",   "Frank", 87000, "2024-06-02")
]
cols = ["id","department","name","salary","hired_date"]
df = spark.createDataFrame(data, cols)

dep_window = Window.partitionBy("department").orderBy(F.desc("salary"), F.desc("hired_date"))
df_with_rn = df.withColumn("rn", F.row_number().over(dep_window))
top3 = df_with_rn.filter(F.col("rn") <= 3).orderBy("department","rn")
top3.show(truncate=False)

df_rank = df.withColumn("rank", F.rank().over(dep_window)) \
            .withColumn("dense_rank", F.dense_rank().over(dep_window)) \
            .orderBy("department", F.desc("salary"), "name")
df_rank.show(truncate=False)

events = [
    (1, "u1", "2024-06-01 10:00", "A"),
    (2, "u1", "2024-06-02 11:00", "B"),
    (3, "u2", "2024-05-30 09:00", "C")
]
events_cols = ["evt_id","user_id","event_time","payload"]
events_df = spark.createDataFrame(events, events_cols) \
                 .withColumn("event_time", F.to_timestamp("event_time"))

w2 = Window.partitionBy("user_id").orderBy(F.desc("event_time"))
df_latest = events_df.withColumn("rn", F.row_number().over(w2)) \
                     .filter(F.col("rn") == 1) \
                     .drop("rn")
df_latest.show(truncate=False)


+---+----------+-----+------+----------+---+
|id |department|name |salary|hired_date|rn |
+---+----------+-----+------+----------+---+
|4  |eng       |Dave |95000 |2024-06-01|1  |
|3  |eng       |Carol|95000 |2024-05-21|2  |
|6  |eng       |Frank|87000 |2024-06-02|3  |
|2  |sales     |Bob  |80000 |2024-06-01|1  |
|5  |sales     |Eve  |80000 |2024-04-15|2  |
|1  |sales     |Alice|70000 |2024-05-01|3  |
+---+----------+-----+------+----------+---+

+---+----------+-----+------+----------+----+----------+
|id |department|name |salary|hired_date|rank|dense_rank|
+---+----------+-----+------+----------+----+----------+
|3  |eng       |Carol|95000 |2024-05-21|2   |2         |
|4  |eng       |Dave |95000 |2024-06-01|1   |1         |
|6  |eng       |Frank|87000 |2024-06-02|3   |3         |
|2  |sales     |Bob  |80000 |2024-06-01|1   |1         |
|5  |sales     |Eve  |80000 |2024-04-15|2   |2         |
|1  |sales     |Alice|70000 |2024-05-01|3   |3         |
+---+----------+-----+------+-------

In [33]:
import time
import pandas as pd
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import pandas_udf

data = [("a", 1), ("bb", 2), ("ccc", 3)] * 200000
df = spark.createDataFrame(data, schema=["s", "n"]).repartition(8)

def length_times_n(s, n):
    if s is None:
        return None
    return len(s) * n

udf_len = F.udf(length_times_n, T.IntegerType())

t0 = time.time()
df_udf = df.withColumn("res", udf_len(F.col("s"), F.col("n")))
df_udf.count()
t_udf = time.time() - t0
print("Tiempo UDF clásico (seg):", t_udf)

@pandas_udf(T.IntegerType())
def length_times_n_vectorized(s: pd.Series, n: pd.Series) -> pd.Series:
    return s.str.len().fillna(0).astype(int) * n

t0 = time.time()
df_pudf = df.withColumn("res", length_times_n_vectorized(F.col("s"), F.col("n")))
df_pudf.count()
t_pudf = time.time() - t0
print("Tiempo Pandas UDF (seg):", t_pudf)
print("Ratio UDF/PandasUDF:", t_udf / t_pudf)

Tiempo UDF clásico (seg): 12.343515396118164
Tiempo Pandas UDF (seg): 12.200708627700806
Ratio UDF/PandasUDF: 1.0117047929571177
